<a href="https://colab.research.google.com/github/amarala931/MIA/blob/main/Entrenamiento_no_supervisado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller Práctico: Aprendizaje No Supervisado con K-Means

¡Bienvenido/a! En este notebook aprenderás a aplicar el algoritmo de agrupamiento **K-Means (K-medias)** utilizando la librería **Scikit-learn**, visualizaciones interactivas con **Plotly** y la descarga del dataset desde la plataforma **Hugging Face**.



## Paso 1: Instalación, carga del dataset desde Hugging Face y conversión a Pandas

### Explicación Teórica y Práctica
Para comenzar, necesitamos instalar y cargar la librería `datasets` de Hugging Face. Hugging Face no solo aloja modelos de lenguaje, sino también miles de conjuntos de datos tabulares, de audio y visión.

Para ello usaremos un dataset llamado [SIH/palmer-penguins](https://huggingface.co/datasets/SIH/palmer-penguins)

Al descargar un dataset con `load_dataset`, obtenemos un objeto llamado `DatasetDict`. Este objeto es un diccionario que contiene distintas particiones (como `train`, `validation` o `test`). En este caso, exploraremos sus particiones e indexaremos la partición principal para convertirla directamente a un `DataFrame` de Pandas usando el método optimizado `.to_pandas()`. Esto nos permitirá utilizar todas las herramientas clásicas de manipulación de datos en Python.

In [ ]:
# Instalar librerías necesarias en el entorno de Colab
!pip install -q datasets plotly pandas scikit-learn

In [ ]:
from datasets import load_dataset
import pandas as pd

# 1. Cargar el dataset 'SIH/palmer-penguins'
ds_dict = load_dataset("SIH/palmer-penguins")

# 2. Mostrar la estructura interna del DatasetDict
print("Estructura del DatasetDict descargado:")
print(ds_dict)

# 3. Extraer la partición disponible (generalmente 'train') y convertir a Pandas DataFrame
particion = list(ds_dict.keys())[0]
df_raw = ds_dict[particion].to_pandas()

# Mostrar las primeras filas del DataFrame
display(df_raw.head())

## Paso 2: Análisis Exploratorio de Datos (EDA) con Plotly

### Explicación Teórica y Práctica
Antes de aplicar cualquier algoritmo, es fundamental entender la distribución, la escala y la calidad de nuestros datos. K-Means es extremadamente sensible a las escalas de las variables (ya que calcula distancias euclidianas) y no puede manejar valores nulos (`NaN`).

Comenzaremos revisando la información general del dataset con `.info()` y `.describe()`. Posteriormente, utilizaremos `plotly.express` para crear un scatter plot interactivo que compare la masa corporal (`body_mass_g`) con la longitud del pico (`bill_length_mm`). Esto nos dará una pista visual de si existen grupos naturales en los datos según su morfología.

In [ ]:
import plotly.express as px

# Información general de los tipos de datos y nulos
print("=== Información general del Dataset ===")
df_raw.info()

# Resumen estadístico de las variables numéricas
print("\n=== Resumen Estadístico ===")
display(df_raw.describe())

# Conteo exacto de valores nulos por columna
print("\n=== Valores Nulos por Columna ===")
print(df_raw.isnull().sum())

In [ ]:
# Crear un gráfico de dispersión interactivo con histogramas marginales usando Plotly
fig = px.scatter(
    df_raw,
    x="bill_length_mm",
    y="body_mass_g",
    color="island",
    marginal_x="histogram",
    marginal_y="box",
    title="Relación entre Longitud del Pico y Masa Corporal por Isla",
    labels={
        "bill_length_mm": "Longitud del Pico (mm)",
        "body_mass_g": "Masa Corporal (g)",
        "island": "Isla"
    },
    template="plotly_white"
)

fig.show()

## Paso 3: Preprocesamiento y adaptación del dataset para Scikit-learn

### Explicación Teórica y Práctica
Para realizar un agrupamiento no supervisado correcto, debemos seguir estos pasos clave:

1. **Eliminar la variable objetivo (`species`)**: Si dejamos la columna `species`, el algoritmo tendrá "fuga de información" y el agrupamiento perdería el sentido de ser no supervisado. Almacenamos estas etiquetas en una variable externa únicamente para contrastar nuestros clústeres al final.
2. **Tratamiento de Nulos**: K-Means de Scikit-learn no acepta valores faltantes. Eliminaremos las filas con valores nulos para garantizar la consistencia matemática.
3. **Codificación y Selección de Variables**: K-Means se basa en distancias. Aunque existen variantes para variables categóricas (como K-Prototypes), el algoritmo estándar funciona óptimamente con variables continuas. Filtraremos y nos quedaremos con las variables numéricas clave: `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm` y `body_mass_g`.
4. **Estandarización (`StandardScaler`)**: K-Means calcula distancias euclidianas:
$$d(x, y) = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}$$
Si una variable tiene una escala en miles (como `body_mass_g` que oscila entre 3000 y 6000) y otra está en decenas (como `bill_depth_mm` que oscila entre 13 y 21), la variable con mayor escala dominará por completo el cálculo de la distancia. La estandarización transforma cada variable para que tenga una media de 0 y una desviación estándar de 1:
$$z = \frac{x - \mu}{\sigma}$$
Esto coloca a todas las variables en igualdad de condiciones.

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Limpieza preliminar: eliminar filas con nulos en las columnas que utilizaremos
columns_to_use = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
df_clean = df_raw.dropna(subset=columns_to_use).copy()

# 2. Guardar las etiquetas reales ('species') por separado para validación final
y_true = df_clean["species"]

# 3. Extraer matriz de características numéricas
X = df_clean[columns_to_use]

# 4. Aplicar estandarización con StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convertir de nuevo a DataFrame para facilitar el análisis posterior
X_scaled_df = pd.DataFrame(X_scaled, columns=columns_to_use)
print("Datos estandarizados con éxito. Dimensiones:", X_scaled_df.shape)
display(X_scaled_df.head())

## Paso 4: Selección del número óptimo de clústeres (K) con Plotly

### Explicación Teórica y Práctica
Determinar el número correcto de grupos ($K$) es uno de los mayores desafíos del aprendizaje no supervisado. Evaluaremos dos métricas para diferentes valores de $K$ (desde 2 hasta 10):

1. **Inercia (Dentro de la Suma de Cuadrados - WCSS)**: Mide qué tan compactos son los clústeres. Buscamos el "Método del Codo" (Elbow Method), donde el decremento de la inercia pasa de ser drástico a ser lineal y suave.
2. **Coeficiente de Silueta**: Mide la calidad del agrupamiento evaluando tanto la cohesión dentro de un clúster como la separación respecto a otros clústeres. Toma valores entre -1 y 1. Un valor más alto y cercano a 1 indica una excelente asignación de clústeres.

In [ ]:
from sklearn.cluster import KMeans

from sklearn.metrics import silhouette_score
import plotly.graph_objects as go

inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    # Inicializamos KMeans. Se usa n_init="auto" por compatibilidad con versiones recientes de sklearn
    kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
    kmeans.fit(X_scaled)

    # Guardar métricas
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# --- Gráfico 1: Método del Codo (Inercia) con Plotly ---
fig_elbow = go.Figure()
fig_elbow.add_trace(go.Scatter(x=list(K_range), y=inertias, mode='lines+markers', line=dict(color='firebrick', width=3)))
fig_elbow.update_layout(
    title="Método del Codo (Inercia vs. K)",
    xaxis_title="Número de Clústeres (K)",
    yaxis_title="Inercia (WCSS)",
    template="plotly_white"
)
fig_elbow.show()

# --- Gráfico 2: Coeficiente de Silueta con Plotly ---
fig_sil = go.Figure()
fig_sil.add_trace(go.Scatter(x=list(K_range), y=silhouette_scores, mode='lines+markers', line=dict(color='royalblue', width=3)))
fig_sil.update_layout(
    title="Coeficiente de Silueta vs. K",
    xaxis_title="Número de Clústeres (K)",
    yaxis_title="Silhouette Score",
    template="plotly_white"
)
fig_sil.show()

## Paso 5: Entrenamiento del modelo K-Means en Scikit-learn

### Explicación Teórica y Práctica
Observando la métrica de Silueta y el Método del Codo, podemos ver que $K=3$ es un punto óptimo muy fuerte (coincidiendo perfectamente con el número de especies reales de pingüinos en el dataset).

Ahora entrenaremos el estimador `KMeans` definitivo con $K=3$ y asignaremos las etiquetas resultantes a nuestro dataset original.

In [ ]:
# Definimos el K óptimo basándonos en los resultados previos
optimal_k = 3

kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init="auto")
kmeans_model.fit(X_scaled)

# Extraer las etiquetas de clúster resultantes
cluster_labels = kmeans_model.labels_

# Añadir las etiquetas al DataFrame original y limpio
df_clean["cluster"] = cluster_labels
df_clean["cluster"] = df_clean["cluster"].astype(str) # Convertir a string para una mejor leyenda visual

print("Asignación de clústeres finalizada de forma exitosa.")
display(df_clean[["species", "cluster"] + columns_to_use].head())

## Paso 6: Visualización de los clústeres e interpretación de resultados

### Explicación Teórica y Práctica
Como nuestros datos originales tienen 4 dimensiones, no es posible visualizarlos directamente en un espacio plano. Utilizaremos **PCA (Análisis de Componentes Principales)** para reducir las dimensiones de 4 a 2 componentes principales que retengan la mayor cantidad de varianza matemática posible.

Finalmente, crearemos una tabla de perfiles promedio y un gráfico de barras interactivo para caracterizar el perfil biológico y físico de cada clúster descubierto.

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

# 1. Aplicar PCA de 2 componentes a los datos escalados
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Crear un DataFrame con las coordenadas de PCA
df_pca = pd.DataFrame(data=X_pca, columns=["PC1", "PC2"])
df_pca["cluster"] = df_clean["cluster"].values
df_pca["species"] = df_clean["species"].values

# Reducir los centroides al mismo espacio dimensional de PCA
centroids_scaled = kmeans_model.cluster_centers_
centroids_pca = pca.transform(centroids_scaled)

# 2. Graficar los datos y los centroides con Plotly
fig_clusters = px.scatter(
    df_pca,
    x="PC1",
    y="PC2",
    color="cluster",
    symbol="species",
    title="Visualización de Clústeres K-Means mediante PCA (2D)",
    labels={"PC1": "Componente Principal 1", "PC2": "Componente Principal 2"},
    template="plotly_white"
)

# Añadir los centroides como puntos rojos grandes en el gráfico
fig_clusters.add_trace(
    go.Scatter(
        x=centroids_pca[:, 0],
        y=centroids_pca[:, 1],
        mode='markers',
        marker=dict(color='Black', size=15, symbol='x', line=dict(width=2)),
        name='Centroides'
    )
)

fig_clusters.show()

In [ ]:
# 3. Tabla Resumen: Promedio de cada variable física según el clúster asignado
resumen_clusters = df_clean.groupby("cluster")[columns_to_use].mean().reset_index()
print("=== Perfil Físico de cada Clúster (Promedios) ===")
display(resumen_clusters)

# 4. Gráfico de barras agrupadas interactivo para comparar las características
df_melt = pd.melt(resumen_clusters, id_vars=['cluster'], value_vars=columns_to_use,
                  var_name='Caracteristica', value_name='Valor_Promedio')

fig_bar = px.bar(
    df_melt,
    x='Caracteristica',
    y='Valor_Promedio',
    color='cluster',
    barmode='group',
    title='Comparativa de Características Físicas por Clúster',
    template='plotly_white'
)
fig_bar.show()